# hallucination-probes: extract features

Generates Qwen2.5-7B answers, last-token hidden states of every layer and BLEURT-20 labels for the chosen
datasets. Save the notebook output as a Kaggle dataset/notebook output and attach it to `kaggle_train.ipynb`.
Resumable: re-running skips finished domains.

In [ ]:
# Clones the repo. Public repo: no credentials needed. If the anonymous clone fails (for example the repo is private
# again) it retries with the GITHUB_TOKEN Kaggle secret, which is passed to git only through an environment header,
# never in the command line or URL, so it cannot show up in tracebacks, logs or the saved clone.
import base64, os, subprocess, urllib.request

REPO, REF = "brainardphilemon/hallucination-probes", "main"   # pin a commit hash for reproducible runs
DEST = "/kaggle/working/hallucination-probes"
URL = f"https://github.com/{REPO}.git"
subprocess.run(["rm", "-rf", DEST], check=True)
clone = subprocess.run(["git", "clone", "-q", URL, DEST], env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
                       capture_output=True, text=True)
if clone.returncode:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Anonymous git clone failed and no GITHUB_TOKEN secret is attached "
                           "(Add-ons -> Secrets). git said: " + clone.stderr.strip()[-300:]) from None
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
           "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader", "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
    subprocess.run(["rm", "-rf", DEST], check=True)
    clone = subprocess.run(["git", "clone", "-q", URL, DEST], env=env, capture_output=True, text=True)
    if clone.returncode:
        try:  # 401 = token invalid/expired; 404 = token cannot see this repo (wrong repository or missing Contents: read)
            request = urllib.request.Request(f"https://api.github.com/repos/{REPO}", headers={"Authorization": f"Bearer {token}"})
            status = urllib.request.urlopen(request).status
        except Exception as exc:
            status = getattr(exc, "code", type(exc).__name__)
        raise RuntimeError(f"git clone failed; GitHub API answered {status} for {REPO} with your token. 401: token invalid or "
                           "expired. 404: the token has no read access to this repository (create it with Resource owner "
                           "brainardphilemon, repository hallucination-probes, Contents: Read-only).") from None
    del token, auth, env
subprocess.run(["git", "-C", DEST, "checkout", "-q", REF], check=True)
print(subprocess.run(["git", "-C", DEST, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r /kaggle/working/hallucination-probes/requirements-kaggle.txt
!pip install -q --no-deps -e /kaggle/working/hallucination-probes

In [ ]:
# ---- config -------------------------------------------------------------------------------
ARTIFACT_NAME = "hprobes_qwen25_7b"          # output folder (= input name for kaggle_train.ipynb)
DOMAINS = ["tqa", "nq_open", "sciq", "popqa", "web_questions", "hotpotqa", "triviaqa"]
SAMPLES_PER_DOMAIN = 0                         # 0 = full dataset
EXTRA_ARGS = []                                # e.g. ["--stage", "generate"], ["--no-4bit"]

In [ ]:
import subprocess, sys
command = [sys.executable, "-m", "hprobes.cli", "extract", "--artifact-dir", f"/kaggle/working/{ARTIFACT_NAME}",
           "--domains", *DOMAINS, "--samples-per-domain", str(SAMPLES_PER_DOMAIN), *EXTRA_ARGS]
print("Running:", " ".join(command))
subprocess.run(command, check=True)